# 22 Rank fund and family features

目标：在当月可投资基金宇宙内按固定方向排名 13 个基金及家族特征。

- 输入：05、06、08 号基金分支输出
- 正式输出：fund_13_ranked_features.parquet、排名宇宙审计
- 主键：wficn+month
- 上游依赖：05、06、08
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import pandas as pd
from src.atomic_write import write_csv_atomic, write_parquet_atomic
from src.ranking import rank_to_half_interval
from src.runtime_paths import output_path, upstream_path

names=['F_ST_Rev', 'F_r2_1', 'F_r12_2', 'age', 'tna', 'flow', 'exp_ratio', 'turnover', 'Family_TNA', 'fund_no', 'Family_r12_2', 'Family_age', 'Family_flow']
basic=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"05_aggregate_shareclasses_and_fund_features","data/processed/fund/fund_month_basic_features.parquet"))
mom=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"06_compute_fund_rret_and_momentum","data/processed/fund/fund_alpha_momentum_raw.parquet"))
fam=pd.read_parquet(upstream_path(ROOT,RUN_MODE,"08_compute_family_features","data/processed/fund/family_features_raw.parquet"))
for x in [basic,mom,fam]: x["month"]=pd.to_datetime(x["month"])
raw=basic.merge(mom[["wficn","month","F_ST_Rev","F_r2_1","F_r12_2"]],on=["wficn","month"],how="left",validate="one_to_one")
raw=raw.merge(fam,on=["wficn","month"],how="left",validate="one_to_one")
directions=CONFIG["feature_directions"]["fund_directions"]; ranked=raw[["wficn","month"]].copy()
for month,idx in raw.groupby("month").groups.items():
    for feature in names:
        ranked.loc[idx,feature]=rank_to_half_interval(raw.loc[idx,feature],direction=int(directions[feature])).to_numpy()
assert not ranked.duplicated(["wficn","month"]).any() and len(names)==13
universe=raw.groupby("month").size().rename("eligible_funds").reset_index()
out=output_path(ROOT,RUN_MODE,"22_rank_fund_and_family_features","data/processed/fund/fund_13_ranked_features.parquet")
aud=output_path(ROOT,RUN_MODE,"22_rank_fund_and_family_features","data/audit/22_fund_ranking_universe.csv")
write_parquet_atomic(ranked,out); write_csv_atomic(universe,aud)
INPUTS_USED=["fund_month_basic_features.parquet","fund_alpha_momentum_raw.parquet","family_features_raw.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud]]
INPUT_COUNTS={"fund_months":len(raw)}; OUTPUT_COUNTS={"ranked":len(ranked)}; PRIMARY_KEY=["wficn","month"]; DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=0
EXCEPTION_FILE=str(aud.relative_to(ROOT)); STAGE_STATUS="passed"; STAGE_NOTES=["Ranking universe is the eligible WFICN-month fund branch.","All 13 directions are the locked fixed values from feature_directions.yaml."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "22_rank_fund_and_family_features", stage="22", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
